In [7]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / ".env")

print("✓ Environment loaded")

✓ Environment loaded


In [8]:
import yfinance as yf
import openpyxl
from openpyxl.styles import Font
from datetime import datetime
import subprocess
import os

print("🔄 Refreshing portfolio tracker...")
print("=" * 40)

# Step 1: Fetch live prices
tickers = {
    "NVDA": None,
    "AMZN": None,
    "BN": None,
    "GOOGL": None,
    "SGOL": None,
    "BMNR": None,
    "PLTR": None,
    "MSTR": None,
}

print("\n📈 Fetching live prices...")
for ticker in tickers:
    data = yf.Ticker(ticker)
    price = round(data.fast_info['last_price'], 2)
    tickers[ticker] = price
    print(f"   {ticker}: ${price}")

btc = yf.Ticker("BTC-USD")
btc_price = round(btc.fast_info['last_price'], 2)
print(f"   BTC: ${btc_price}")

# Step 2: Update Excel
file_path = r'C:\Users\13185\claude-cookbooks\skills\notebooks\portfolio_tracker.xlsx'
wb = openpyxl.load_workbook(file_path)
ws = wb.active

# Update timestamp in A1
ws['A1'] = 'Last Updated:'
ws['B1'] = datetime.now().strftime('%B %d, %Y at %I:%M %p')
ws['A1'].font = Font(bold=True, size=11)
ws['B1'].font = Font(size=11)

# Update current prices
for row in ws.iter_rows():
    for cell in row:
        if cell.column == 1 and cell.value in tickers:
            price_cell = ws.cell(row=cell.row, column=4)
            price_cell.value = tickers[cell.value]
        elif cell.column == 1 and cell.value == "BTC":
            price_cell = ws.cell(row=cell.row, column=4)
            price_cell.value = btc_price

wb.save(file_path)
print("\n✅ Portfolio tracker updated!")
print(f"   Timestamp: {datetime.now().strftime('%B %d, %Y at %I:%M %p')}")

# Step 3: Open the file
subprocess.Popen(file_path, shell=True)
print("\n📂 Opening portfolio_tracker.xlsx...")

🔄 Refreshing portfolio tracker...

📈 Fetching live prices...
   NVDA: $223.47
   AMZN: $265.01
   BN: $45.34
   GOOGL: $388.91
   SGOL: $43.3
   BMNR: $19.39
   PLTR: $137.15
   MSTR: $165.81
   BTC: $77552.99

✅ Portfolio tracker updated!
   Timestamp: May 20, 2026 at 07:20 PM

📂 Opening portfolio_tracker.xlsx...


In [10]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import matplotlib.pyplot as plt
import io

print("📊 Generating portfolio summary presentation...")

# Portfolio data using prices already fetched
holdings = {
    "NVDA":  {"shares": 2.75,  "avg_cost": 112.69,  "current_price": tickers["NVDA"]},
    "AMZN":  {"shares": 1.53,  "avg_cost": 227.70,  "current_price": tickers["AMZN"]},
    "BN":    {"shares": 6.0,   "avg_cost": 41.63,   "current_price": tickers["BN"]},
    "GOOGL": {"shares": 0.025, "avg_cost": 385.11,  "current_price": tickers["GOOGL"]},
    "SGOL":  {"shares": 1.7,   "avg_cost": 46.24,   "current_price": tickers["SGOL"]},
    "BMNR":  {"shares": 3.5,   "avg_cost": 21.42,   "current_price": tickers["BMNR"]},
    "PLTR":  {"shares": 0.94,  "avg_cost": 159.00,  "current_price": tickers["PLTR"]},
    "MSTR":  {"shares": 9.75,  "avg_cost": 254.93,  "current_price": tickers["MSTR"]},
    "BTC":   {"shares": 0.016,  "avg_cost": 85346.00, "current_price": btc_price},
}

# Calculate values
for ticker, data in holdings.items():
    data["cost_basis"] = round(data["shares"] * data["avg_cost"], 2)
    data["current_value"] = round(data["shares"] * data["current_price"], 2)
    data["gain_loss"] = round(data["current_value"] - data["cost_basis"], 2)
    data["gain_loss_pct"] = round((data["gain_loss"] / data["cost_basis"]) * 100, 2)

total_cost = sum(d["cost_basis"] for d in holdings.values())
total_value = sum(d["current_value"] for d in holdings.values())
total_gain_loss = round(total_value - total_cost, 2)
total_gain_loss_pct = round((total_gain_loss / total_cost) * 100, 2)

print(f"   Total Cost Basis: ${total_cost:,.2f}")
print(f"   Total Value:      ${total_value:,.2f}")
print(f"   Total Gain/Loss:  ${total_gain_loss:,.2f} ({total_gain_loss_pct}%)")
print("\n✓ Data calculated")

📊 Generating portfolio summary presentation...
   Total Cost Basis: $5,071.84
   Total Value:      $4,429.67
   Total Gain/Loss:  $-642.17 (-12.66%)

✓ Data calculated


In [ ]:
prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

# ── SLIDE 1: Portfolio Overview ──
slide1 = prs.slides.add_slide(prs.slide_layouts[6])  # blank

def add_textbox(slide, text, x, y, w, h, fontsize=18, bold=False, color=RGBColor(0,0,0), align=PP_ALIGN.LEFT):
    txBox = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = txBox.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(fontsize)
    run.font.bold = bold
    run.font.color.rgb = color

# Background header bar
from pptx.util import Emu
header = slide1.shapes.add_shape(1, Inches(0), Inches(0), Inches(13.33), Inches(1.3))
header.fill.solid()
header.fill.fore_color.rgb = RGBColor(0x1F, 0x4E, 0x79)
header.line.fill.background()

add_textbox(slide1, "Portfolio Summary", 0.3, 0.15, 8, 1, fontsize=32, bold=True, color=RGBColor(255,255,255))
add_textbox(slide1, datetime.now().strftime('%B %d, %Y'), 9.5, 0.25, 3.5, 0.8, fontsize=18, color=RGBColor(255,255,255), align=PP_ALIGN.RIGHT)

# Summary stats
gl_color = RGBColor(0x00, 0x70, 0x00) if total_gain_loss >= 0 else RGBColor(0xC0, 0x00, 0x00)
add_textbox(slide1, f"Total Value", 0.5, 1.5, 4, 0.5, fontsize=14, bold=True, color=RGBColor(0x1F,0x4E,0x79))
add_textbox(slide1, f"${total_value:,.2f}", 0.5, 1.95, 4, 0.7, fontsize=28, bold=True)
add_textbox(slide1, f"Total Cost Basis", 4.5, 1.5, 4, 0.5, fontsize=14, bold=True, color=RGBColor(0x1F,0x4E,0x79))
add_textbox(slide1, f"${total_cost:,.2f}", 4.5, 1.95, 4, 0.7, fontsize=28, bold=True)
add_textbox(slide1, f"Total Gain/Loss", 8.5, 1.5, 4.5, 0.5, fontsize=14, bold=True, color=RGBColor(0x1F,0x4E,0x79))
add_textbox(slide1, f"${total_gain_loss:,.2f} ({total_gain_loss_pct}%)", 8.5, 1.95, 4.5, 0.7, fontsize=28, bold=True, color=gl_color)

# Holdings table
headers = ["Ticker", "Shares", "Avg Cost", "Current Price", "Cost Basis", "Value", "Gain/Loss $", "Gain/Loss %"]
col_positions = [0.3, 1.6, 2.7, 4.0, 5.3, 6.6, 7.9, 9.5]
col_widths =    [1.2, 1.0, 1.2, 1.2, 1.2, 1.2, 1.2, 1.5]

y_start = 2.9
for i, h in enumerate(headers):
    add_textbox(slide1, h, col_positions[i], y_start, col_widths[i], 0.35, fontsize=10, bold=True, color=RGBColor(0x1F,0x4E,0x79))

for row_idx, (ticker, data) in enumerate(holdings.items()):
    y = y_start + 0.38 + (row_idx * 0.38)
    gl_c = RGBColor(0x00,0x70,0x00) if data["gain_loss"] >= 0 else RGBColor(0xC0,0x00,0x00)
    row_data = [
        ticker,
        f"{data['shares']}",
        f"${data['avg_cost']:,.2f}",
        f"${data['current_price']:,.2f}",
        f"${data['cost_basis']:,.2f}",
        f"${data['current_value']:,.2f}",
        f"${data['gain_loss']:,.2f}",
        f"{data['gain_loss_pct']}%",
    ]
    for i, val in enumerate(row_data):
        color = gl_c if i >= 6 else RGBColor(0,0,0)
        add_textbox(slide1, val, col_positions[i], y, col_widths[i], 0.35, fontsize=10, color=color)

# ── SLIDE 2: Portfolio Allocation Chart ──
slide2 = prs.slides.add_slide(prs.slide_layouts[6])
header2 = slide2.shapes.add_shape(1, Inches(0), Inches(0), Inches(13.33), Inches(1.3))
header2.fill.solid()
header2.fill.fore_color.rgb = RGBColor(0x1F, 0x4E, 0x79)
header2.line.fill.background()
add_textbox(slide2, "Portfolio Allocation", 0.3, 0.15, 8, 1, fontsize=32, bold=True, color=RGBColor(255,255,255))

labels = list(holdings.keys())
values = [holdings[t]["current_value"] for t in labels]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#C00000' if v == max(values) else '#1F4E79' for v in values]
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Portfolio Allocation by Current Value', fontsize=14, fontweight='bold')
ax.set_xlabel('Position', fontsize=11)
ax.set_ylabel('Current Value ($)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()

img_stream = io.BytesIO()
plt.savefig(img_stream, format='png', dpi=150, bbox_inches='tight')
img_stream.seek(0)
plt.close()

slide2.shapes.add_picture(img_stream, Inches(1.5), Inches(1.4), Inches(10), Inches(5.5))

# Save
pptx_path = r'C:\Users\13185\claude-cookbooks\skills\notebooks\portfolio_summary.pptx'
prs.save(pptx_path)
print("✅ PowerPoint saved!")

subprocess.Popen(pptx_path, shell=True)
print("📂 Opening portfolio_summary.pptx...")

✅ PowerPoint saved!
📂 Opening portfolio_summary.pptx...
